# 01 - Decision Layer Cookbook

This notebook walks through the core Metatate flow:

1. discover governed tables
2. inspect business and column meaning
3. authorize a proposed use
4. validate generated SQL
5. explain a decision


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


## 1. Discover governed tables


In [ ]:
discover = client.discover_context(database="ACMECLOUD_DEMO", schema="PUBLIC")
pd.DataFrame(discover["data"]["tables"])[
    ["full_table_name", "sensitivity", "enforcement_mode", "has_pii", "domain", "owner"]
]


## 2. Get table-level decision context


In [ ]:
table_name = "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS"
context = client.get_decision_context(table_name)
print(json.dumps(context["data"]["policy_summary"], indent=2))
print(json.dumps(context["data"]["business_context"], indent=2))


## 3. Inspect column meaning


In [ ]:
meaning = client.inspect_data_meaning(table_name)
pd.DataFrame(meaning["data"]["columns"])[
    ["column_name", "data_type_label", "data_category", "is_pii", "effective_sensitivity", "masking_type"]
]


## 4. Authorize analytics


In [ ]:
analytics = client.authorize_use(
    table_name,
    operation="read",
    intended_use="analytics",
    actor_role="DATA_ANALYST",
    columns=["CUSTOMER_ID", "EMAIL", "ARR"],
)
print(json.dumps(analytics["data"], indent=2))


## 5. Validate generated SQL before execution


In [ ]:
sql = "SELECT customer_id, email, arr FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE region = 'EU'"
validation = client.validate_query_context(
    sql,
    operation="read",
    intended_use="analytics",
    actor_role="DATA_ANALYST",
)
print(json.dumps(validation["data"], indent=2))
